In [1]:
import sys; sys.path.insert(0, "../scripts")   # wherever inspect_panel.py lives
import polars as pl
import inspect_panel as ip

pl.Config(tbl_rows=50, tbl_cols=20, fmt_str_lengths=60)
p = ip.load("../panel")
p.height, p["TDLinx"].n_unique(), p["year"].unique().sort().to_list()

(571002,
 60832,
 [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

In [13]:
p.group_by("Gruppe").len().sort("len", descending=True)

Gruppe,len
str,u32
"""Edeka""",142369
"""Rewe""",89028
"""Aldi""",50179
"""Schwarz""",47620
"""Rossmann""",26056
"""dm""",23824
null,22939
"""Norma""",15863
"""Müller""",6934


In [5]:
matrix_supermarkets = pl.read_parquet("/home/l/Projects/ZEW/edeka/data/catchments/solve/de_100m/supermarkt")

In [8]:
matrix_supermarkets

cell_id,pop,store_idx,euclid_m,durations,certified
i64,f32,"array[i32, 165]","array[f32, 165]","array[f32, 165]",bool
0,2.301776,"[12698, 3423, … 0]","[4482.411133, 7405.452637, … 20557.123047]","[304.799988, 571.599976, … inf]",true
1,20.761648,"[12698, 3423, … 0]","[3362.748535, 6300.915527, … 19565.851562]","[304.0, 570.799988, … inf]",true
2,19.151192,"[12698, 3423, … 0]","[3328.080322, 6255.590332, … 19485.185547]","[295.600006, 562.400024, … inf]",true
3,0.474018,"[12698, 3423, … 0]","[2945.201172, 5887.571777, … 19189.041016]","[258.0, 524.799988, … inf]",true
4,19.173784,"[12698, 3423, … 0]","[3279.942383, 6204.151855, … 19426.285156]","[286.399994, 553.200012, … inf]",true
5,7.274783,"[12698, 3423, … 0]","[3011.504639, 5949.387695, … 19230.746094]","[245.100006, 511.899994, … inf]",true
6,9.643689,"[12698, 3423, … 0]","[3303.208008, 6221.480957, … 19424.839844]","[300.299988, 567.099976, … inf]",true
7,19.711733,"[12698, 3423, … 0]","[2980.719238, 5916.101074, … 19190.388672]","[233.800003, 500.600006, … inf]",true
8,6.027814,"[12698, 3423, … 0]","[1712.615723, 4576.583008, … 19094.541016]","[175.600006, 442.399994, … inf]",true


In [ ]:
p.columns

In [ ]:
ip.keys(p) # Duplicate (TDLinx, year) rows

In [ ]:
ip.coverage(p).show(None) #Store counts per year × Shop Type, and the share of nulls per column per year. These show coding breaks between years.
ip.missing(p).show(None)

In [ ]:
ip.numerics(p).filter(pl.col("var") == "VKFL") # Per year: nulls, zeros, negatives, p1/p50/p99 and max for VKFL, the Umsatz columns, Parkplätze and Kassen

In [ ]:
ip.components(p) # Rows where Food + Nonfood differs from the total by more than 1%, for both VKFL and Umsatzpotential

In [ ]:
ip.productivity(p).sort("z") # Stores whose log(€/m²) is more than 4 robust SDs away from the median for their Shop Type × year. This usually catches unit errors in VKFL.

In [ ]:
bad, shared, moves = ip.coords(p); bad # Missing or out-of-Germany lat/lon, several stores on one coordinate in the same year, and moves of more than 500 m between years

In [ ]:

adm_rows, adm_multi = ip.admin(p); adm_rows

In [ ]:
sw = ip.switchers(p); sw.filter("flip_back")

In [ ]:
ip.switch_flows(p, "Gruppe")

In [ ]:
ip.gaps(p)

In [ ]:
tot, jumps = ip.totals(p); jumps

In [ ]:
def store(tdlinx, cols=("year", "Name", "Banner", "Gruppe", "Gruppe_src", "Chain", "Shop Type",
                        "VKFL", ip.U, "lat", "lon", "full_address")):
    return p.filter(pl.col("TDLinx") == tdlinx).select(cols)

store(sw.filter("flip_back")["TDLinx"][0])

In [ ]:
p.group_by(
    pl.col("Name"),
    # pl.col("Name-juristisch"),
    # pl.col("Banner"),
    # pl.col("Handelszentrale"),
).len().sort(pl.col("len"), descending = True).show(None)

In [ ]:
p.filter(pl.col("Name")=="Rew City")

In [ ]:
p.group_by(pl.col("Gruppe")).len().sort(pl.col("len"), descending=True).show(None)

In [ ]:

p.filter(
    pl.col("Gruppe").is_null(),
    pl.col("Handelszentrale").is_not_null()
    ).select(
         "Name",
         "Name-juristisch",
         "Handelszentrale",
         "Gruppe"
     ).unique(pl.col("Handelszentrale")).sort("Handelszentrale", descending=False)

In [ ]:
cols = ["Name", "Name-juristisch", "Handelszentrale", "Gruppe", "Chain"]

with pl.Config(tbl_rows=-1):
    p.group_by("year").agg(pl.col(cols).null_count(), pl.len().alias("rows")).sort("year").show(None)

In [ ]:
# 1. what the fix removed, per year
p.filter(pl.col("Gruppe_src") == "supplied").group_by("year").agg(
    pl.len(), pl.col("VKFL").sum()).sort("year")


In [ ]:
# 2. are the removed stores still in the panel from 2020? (false = closed before 2020)
later = p.filter(pl.col("year") >= 2020).select("TDLinx").unique().with_columns(pl.lit(True).alias("in_2020+"))
(p.filter(pl.col("Gruppe_src") == "supplied")
   .join(later, on="TDLinx", how="left")
   .group_by("year", pl.col("in_2020+").fill_null(False)).len().sort("year"))


In [ ]:
# 3. what really changed in 2020: buying-centre values of independent stores
(p.filter(pl.col("Betriebsform zur Handelszentrale") == "Selbständig", pl.col("year").is_in([2019, 2020]))
   .group_by("year", "Handelszentrale").len()
   .sort("len", descending=True).head(30))